# Finding 2 — The Northeast Leapfrog
**Study:** Mapping UPI: Scale, Geography, and Competition in India's Payment Ecosystem
**Data:** NPCI State-wise Statistics (2023–2026)
**Analysis:** State-wise CAGR (2023–2026), inclusion signal via Pearson correlation, Northeast vs non-Northeast growth comparison

In [ ]:
import subprocess, sys

for pkg in ["matplotlib", "seaborn", "pandas", "numpy", "sqlalchemy", "pymysql"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from sqlalchemy import create_engine
from pathlib import Path

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.6)
plt.rcParams.update({
    "figure.dpi": 200,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("All dependencies ready.")

In [ ]:
# MySQL connection — set your own credentials via environment or edit below.
DB_USER = "your_username"    # your MySQL username
DB_PASS = "your_password"    # your MySQL password
DB_HOST = "localhost"        # or your MySQL host
DB_NAME = "upi_data"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}?charset=utf8mb4"
)

# State-wise volume — NPCI Ecosystem Statistics 2023–2026
raw = pd.read_sql("""
    SELECT month_year, state, volume_mn, value_cr
    FROM upi_statewise
    ORDER BY month_year, state
""", engine)

df = raw.copy()

# Strip trailing '#' and stray spaces
df["state"] = df["state"].str.rstrip("# ").str.strip()

# Remove " Total" suffix that NPCI appends to subtotal rows
df["state"] = df["state"].str.replace(" Total", "", regex=False).str.strip()
df["state"] = df["state"].str.replace("Total", "", regex=False).str.strip()

# Drop catch-all and summary rows
bad = (
    df["state"].str.lower().str.contains("unclassified", na=True) |
    df["state"].str.lower().str.contains("all india", na=True) |
    df["state"].str.lower().str.contains("others", na=True) |
    df["state"].str.contains("#", na=True) |
    df["state"].str.strip().str.lower().isin(["total", "unknown", ""])
)
df = df[~bad].copy()

# Title case normalisation
df["state"] = df["state"].str.title()

# Drop null / zero volumes
df = df.dropna(subset=["volume_mn"])
df = df[df["volume_mn"] > 0].copy()

df["date"] = pd.to_datetime(df["month_year"] + "-01")
df = df.sort_values(["month_year", "state"]).reset_index(drop=True)

print(f"Raw rows  : {len(raw)}")
print(f"Clean rows: {len(df)}")
print(f"Months    : {df['month_year'].nunique()}  ({df['month_year'].min()} -> {df['month_year'].max()})")
print(f"States    : {df['state'].nunique()}")
print(f"\nAll states ({df['state'].nunique()}):")
for s in sorted(df["state"].unique()):
    print(f"  {s}")

In [ ]:
first_month = df["month_year"].min()
last_month  = df["month_year"].max()

first_date = pd.to_datetime(first_month + "-01")
last_date  = pd.to_datetime(last_month  + "-01")
n_years    = (last_date - first_date).days / 365.25

first_vol_s = df[df["month_year"] == first_month].groupby("state")["volume_mn"].sum()
last_vol_s  = df[df["month_year"] == last_month].groupby("state")["volume_mn"].sum()

common = first_vol_s.index.intersection(last_vol_s.index)
cagr_df = pd.DataFrame({
    "state":     common,
    "first_vol": first_vol_s[common].values,
    "last_vol":  last_vol_s[common].values,
})
cagr_df = cagr_df[cagr_df["first_vol"] > 0].copy()
cagr_df["cagr_pct"] = (
    (cagr_df["last_vol"] / cagr_df["first_vol"]) ** (1 / n_years) - 1
) * 100
cagr_df = cagr_df.sort_values("cagr_pct", ascending=False).reset_index(drop=True)

# Asterisk label for small-base states 
SMALL_BASE = 1   # Mn
cagr_df["label"] = cagr_df.apply(
    lambda r: r["state"] + "*" if r["first_vol"] < SMALL_BASE else r["state"], axis=1
)

nat_avg_cagr = cagr_df["cagr_pct"].mean()

# Northeast states (historically underbanked) 
NE_STATES = {
    "Mizoram", "Meghalaya", "Nagaland", "Arunachal Pradesh",
    "Tripura", "Assam", "Manipur", "Sikkim",
    "Jammu And Kashmir", "Ladakh",
}

# Shared footnote for all three charts
CHART_FOOTNOTE = (
    f"* Union territories / states with starting volume below {SMALL_BASE} Mn "
    "— interpret CAGR with caution (small base effect)"
)

ne_mask = cagr_df["state"].isin(NE_STATES)
print(f"Period  : {first_month} -> {last_month}  ({n_years:.2f} years)")
print(f"States  : {len(cagr_df)}  | national avg CAGR: {nat_avg_cagr:.1f}%")
print(f"NE in data : {cagr_df[ne_mask]['state'].tolist()}")
print(f"Small-base : {cagr_df[cagr_df['first_vol'] < SMALL_BASE]['state'].tolist()}")
print()
print(cagr_df[["label", "first_vol", "last_vol", "cagr_pct"]].to_string(index=False))

## Chart 1 — Growth vs Starting Base (Scatter)

In [ ]:
try:
    from adjustText import adjust_text
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "adjustText", "-q"])
    from adjustText import adjust_text

ne_mask = cagr_df["state"].isin(NE_STATES)

def dot_color(row):
    if row["cagr_pct"] >= nat_avg_cagr:
        return "#16a34a"   # green — above avg
    else:
        return "#dc2626"   # red   — below avg

dot_colors = cagr_df.apply(dot_color, axis=1).values

fig, ax = plt.subplots(figsize=(22, 15))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

ax.scatter(cagr_df["first_vol"], cagr_df["cagr_pct"],
           c=dot_colors, s=180, alpha=0.90, zorder=4,
           edgecolors="white", linewidths=1.5)

# Set limits before text so adjustText has correct axes
ax.set_xscale("log")
ax.set_xlim(cagr_df["first_vol"].min() * 0.4,
            cagr_df["first_vol"].max() * 3.0)
ax.set_ylim(cagr_df["cagr_pct"].min() - 10,
            cagr_df["cagr_pct"].max() + 16)

# Reference lines
ax.axhline(nat_avg_cagr, color="#6366f1", lw=2.0, ls="--", alpha=0.70,
           label=f"National avg CAGR: {nat_avg_cagr:.1f}%")
ax.axvline(cagr_df["first_vol"].median(), color="#94a3b8",
           lw=0.8, ls=":", alpha=0.40, zorder=1)

# Labels for ALL states — NE states in bold, others normal weight
texts = []
for _, row in cagr_df.iterrows():
    ink = dot_color(row)
    is_ne = row["state"] in NE_STATES
    t = ax.text(
        row["first_vol"], row["cagr_pct"], row["label"],
        fontsize=12, color=ink, zorder=5,
        fontweight="bold" if is_ne else "normal",
        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.65)
    )
    texts.append(t)

try:
    adjust_text(texts, ax=ax,
                arrowprops=dict(arrowstyle="-", color="#b0b8c4", lw=0.5))
except Exception as e:
    print(f"adjustText: {e}")

# Northeast dominance annotation (top-left)
ax.text(0.013, 0.985,
        "Northeast states dominate this quadrant —\n8 of top 10 fastest growing states",
        transform=ax.transAxes, fontsize=14, va="top", ha="left",
        color="#166534", style="italic",
        bbox=dict(boxstyle="round,pad=0.55", fc="#f0fdf4", ec="#16a34a",
                  alpha=0.94, lw=1.2))

# Leaders Pulling Ahead (bottom-right)
ax.text(0.987, 0.022, "Low growth, High base\n(Leaders Pulling Ahead)",
        transform=ax.transAxes, fontsize=13, va="bottom", ha="right",
        color="#dc2626", style="italic",
        bbox=dict(boxstyle="round,pad=0.45", fc="#fef2f2", ec="#dc2626",
                  alpha=0.88, lw=1.0))

# Legend
above_n = int((cagr_df["cagr_pct"] >= nat_avg_cagr).sum())
below_n = int((cagr_df["cagr_pct"] <  nat_avg_cagr).sum())
green_p = mpatches.Patch(color="#16a34a", label=f"Above avg CAGR  (n={above_n})")
red_p   = mpatches.Patch(color="#dc2626", label=f"Below avg CAGR  (n={below_n})")
leg_h, leg_l = ax.get_legend_handles_labels()
ax.legend(handles=[green_p, red_p] + leg_h,
          fontsize=13, framealpha=0.96, edgecolor="#e2e8f0", loc="upper right")

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:g}"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.tick_params(labelsize=14)
ax.set_xlabel("Starting Volume (Mn transactions, log scale)", fontsize=16)
ax.set_ylabel("CAGR % per year", fontsize=16)
ax.set_title("UPI Growth vs Starting Base — The Northeast Leapfrog",
             fontsize=20, fontweight="bold", pad=16)
ax.grid(axis="both", alpha=0.14, lw=0.6)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT / "state_cagr_scatter.png", bbox_inches="tight")
plt.show()
print(f"Saved -> output/state_cagr_scatter.png  ({len(cagr_df)} states)")
ne_above = int((ne_mask & (cagr_df["cagr_pct"] >= nat_avg_cagr)).sum())
print(f"NE states above national avg: {ne_above} / {int(ne_mask.sum())}")

## Chart 2 — CAGR by State (Horizontal Bar)

In [ ]:
sorted_cagr = cagr_df.sort_values("cagr_pct", ascending=True).reset_index(drop=True)
n = len(sorted_cagr)

# Top 5 → green, bottom 5 → red, others → grey
bar_colors = []
for i in range(n):
    rank_from_top = n - 1 - i
    if rank_from_top < 5:
        bar_colors.append("#16a34a")
    elif i < 5:
        bar_colors.append("#dc2626")
    else:
        bar_colors.append("#94a3b8")

fig_h = max(10, n * 0.44)
fig, ax = plt.subplots(figsize=(20, fig_h))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

bars = ax.barh(sorted_cagr["label"], sorted_cagr["cagr_pct"],
               color=bar_colors, alpha=0.88, height=0.70)

x_max = sorted_cagr["cagr_pct"].abs().max()
for bar, val in zip(bars, sorted_cagr["cagr_pct"]):
    ax.text(
        bar.get_width() + x_max * 0.012,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}%",
        va="center", fontsize=13, color="#374151"
    )

ax.axvline(nat_avg_cagr, color="#6366f1", lw=2.0, ls="--", alpha=0.75,
           label=f"National avg: {nat_avg_cagr:.1f}%")

green_p = mpatches.Patch(color="#16a34a", alpha=0.88, label="Top 5 fastest")
grey_p  = mpatches.Patch(color="#94a3b8", alpha=0.88, label="Others")
red_p   = mpatches.Patch(color="#dc2626", alpha=0.88, label="Bottom 5 slowest")
leg_h, _ = ax.get_legend_handles_labels()
ax.legend(handles=[green_p, grey_p, red_p] + leg_h,
          fontsize=13, framealpha=0.95, edgecolor="#e5e7eb", loc="lower right")

ax.set_xlabel("CAGR (% per year)", fontsize=16)
ax.set_title(
    f"State-wise UPI Transaction Growth Rate (CAGR {first_month[:4]}–{last_month[:4]})",
    fontsize=20, fontweight="bold", pad=16
)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.tick_params(labelsize=14)
ax.grid(axis="x", alpha=0.25, lw=0.6)
ax.grid(axis="y", alpha=0)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT / "state_cagr_bar.png", bbox_inches="tight")
plt.show()
print(f"Saved -> output/state_cagr_bar.png  ({n} states)")

## Chart 3 — Volume Trajectory: Leaders vs Fast Growers

In [ ]:
# Top 3 by ending volume (solid) vs top 3 by CAGR not already in vol group (dashed)
vol_leaders  = cagr_df.nlargest(3, "last_vol")["state"].tolist()
cagr_leaders = [s for s in cagr_df.nlargest(6, "cagr_pct")["state"]
                if s not in vol_leaders][:3]

PALETTE_VOL  = ["#1e3a5f", "#16a34a", "#dc2626"]
PALETTE_CAGR = ["#7c3aed", "#b45309", "#0891b2"]

VOL_LABEL_OFFSET = [10, -12, 0]

print("Volume leaders :", vol_leaders)
print("Fast growers   :", cagr_leaders)

traj_vol  = df[df["state"].isin(vol_leaders)].sort_values(["state", "date"])
traj_cagr = df[df["state"].isin(cagr_leaders)].sort_values(["state", "date"])

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(22, 15),
    gridspec_kw={"height_ratios": [3, 2], "hspace": 0.40}
)
fig.patch.set_facecolor("white")

# TOP PANEL: Volume Leaders
ax_top.set_facecolor("white")
for i, state in enumerate(vol_leaders):
    sdf = traj_vol[traj_vol["state"] == state].sort_values("date")
    ax_top.plot(sdf["date"], sdf["volume_mn"],
                color=PALETTE_VOL[i], lw=3.2, ls="-",
                label=f"{state}  (Vol Leader)")
    last_r = sdf.iloc[-1]
    ax_top.annotate(state,
                    xy=(last_r["date"], last_r["volume_mn"]),
                    xytext=(7, VOL_LABEL_OFFSET[i]),
                    textcoords="offset points",
                    fontsize=14, color=PALETTE_VOL[i], va="center", fontweight="bold")

ax_top.set_xlim(traj_vol["date"].min(), traj_vol["date"].max() + pd.DateOffset(months=5))
ax_top.set_ylim(0)
ax_top.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax_top.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax_top.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:,.0f}"))
ax_top.tick_params(axis="x", rotation=30, labelsize=14)
ax_top.tick_params(axis="y", labelsize=14)
ax_top.set_ylabel("Monthly Volume (Mn)", fontsize=15)
ax_top.set_title(
    f"Volume Leaders vs Fast Growers — UPI Trajectory {first_month[:4]}–{last_month[:4]}",
    fontsize=20, fontweight="bold", pad=14
)
ax_top.legend(fontsize=13, framealpha=0.95, edgecolor="#e5e7eb", loc="upper left")
ax_top.grid(axis="y", alpha=0.25, lw=0.6)
ax_top.grid(axis="x", alpha=0)
ax_top.spines[["top", "right"]].set_visible(False)

# BOTTOM PANEL: Fast Growers
ax_bot.set_facecolor("white")
for i, state in enumerate(cagr_leaders):
    sdf = traj_cagr[traj_cagr["state"] == state].sort_values("date")
    ax_bot.plot(sdf["date"], sdf["volume_mn"],
                color=PALETTE_CAGR[i], lw=2.8, ls="--",
                label=f"{state}  (Fast Grower)")
    last_r = sdf.iloc[-1]
    ax_bot.annotate(state,
                    xy=(last_r["date"], last_r["volume_mn"]),
                    xytext=(6, 0), textcoords="offset points",
                    fontsize=14, color=PALETTE_CAGR[i], va="center", fontweight="bold")

ax_bot.set_xlim(traj_cagr["date"].min(), traj_cagr["date"].max() + pd.DateOffset(months=5))
ax_bot.set_ylim(0)
ax_bot.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax_bot.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax_bot.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:,.2f}"))
ax_bot.tick_params(axis="x", rotation=30, labelsize=14)
ax_bot.tick_params(axis="y", labelsize=14)
ax_bot.set_ylabel("Monthly Volume (Mn)", fontsize=15)
ax_bot.legend(fontsize=13, framealpha=0.95, edgecolor="#e5e7eb", loc="upper left")
ax_bot.grid(axis="y", alpha=0.25, lw=0.6)
ax_bot.grid(axis="x", alpha=0)
ax_bot.spines[["top", "right"]].set_visible(False)

plt.savefig(OUTPUT / "volume_trajectory.png", bbox_inches="tight")
plt.show()
print("Saved -> output/volume_trajectory.png")

## Summary Statistics

In [ ]:
ne_mask     = cagr_df["state"].isin(NE_STATES)
ne_avg_cagr = cagr_df[ne_mask]["cagr_pct"].mean()
non_ne_avg  = cagr_df[~ne_mask]["cagr_pct"].mean()
ne_gap      = ne_avg_cagr - non_ne_avg

# Pearson correlation: log(starting volume) vs CAGR
corr = float(np.corrcoef(np.log(cagr_df["first_vol"]), cagr_df["cagr_pct"])[0, 1])

if corr < -0.3:
    divide_stmt = (
        "Negative correlation — low-base states growing faster.\n"
        "  Digital divide is CLOSING: UPI is enabling financial inclusion."
    )
elif corr > 0.3:
    divide_stmt = (
        "Positive correlation — high-base states growing faster.\n"
        "  Digital divide is WIDENING: rich states accelerating ahead."
    )
else:
    divide_stmt = (
        "Near-zero correlation — growth rate independent of base size.\n"
        "  No strong convergence or divergence signal."
    )

top5_ne  = cagr_df[ne_mask].nlargest(5, "cagr_pct")
top5_vol = cagr_df.nlargest(5, "last_vol")

print("=" * 66)
print("  STATE-WISE UPI GROWTH — KEY FINDINGS")
print("=" * 66)
print(f"\n  Period             : {first_month} -> {last_month}  ({n_years:.1f} years)")
print(f"  States in analysis : {len(cagr_df)}")
print(f"  National avg CAGR  : {nat_avg_cagr:.1f}% per year")

print(f"\n NORTHEAST VS REST")
print(f"  NE avg CAGR        : {ne_avg_cagr:.1f}% / yr  (n={int(ne_mask.sum())} states)")
print(f"  Non-NE avg CAGR    : {non_ne_avg:.1f}% / yr")
print(f"  Gap (NE advantage) : +{ne_gap:.1f} percentage points per year")

print(f"\n  Top NE states by CAGR:")
for _, r in top5_ne.iterrows():
    flag = "*" if r["first_vol"] < SMALL_BASE else ""
    print(f"    {r['state']:<30} {r['cagr_pct']:+.1f}% / yr  (start: {r['first_vol']:.2f} Mn){flag}")

print(f"\n  VOLUME LEADERS (by ending volume)")
for _, r in top5_vol.iterrows():
    ne_tag = " [NE]" if r["state"] in NE_STATES else ""
    print(f"    {r['state']:<30} {r['last_vol']:>8,.1f} Mn  (CAGR: {r['cagr_pct']:+.1f}%){ne_tag}")

print(f"\n  DIGITAL DIVIDE SIGNAL")
print(f"  Corr (log base vol vs CAGR): r = {corr:.3f}")
print(f"  -> {divide_stmt}")
print("=" * 66)